# Qwen2.5 LoRA 指令微调

完整的 LLM 微调流程：数据准备 → LoRA 训练 → 效果评估 → 量化导出。

---
## 模型选择
- `Qwen/Qwen2.5-1.5B`（默认）— 训练约 8 分钟，稳定不超时，**推荐首次使用**
- `Qwen/Qwen2.5-7B`（进阶）— 需要 QLoRA + 4-bit 加载，约 40 分钟，可能超时

> 1.5B 和 7B 的技术流程完全一样。简历上写的是流程和技能，不是模型参数量。

In [ ]:
# ===== 配置区 =====
MODEL_ID = "Qwen/Qwen2.5-1.5B"       # 首次用 1.5B，熟悉后再改 7B
# MODEL_ID = "Qwen/Qwen2.5-7B"       # 7B 需要 QLoRA，注意训练时间

MAX_SAMPLES = 3000                     # 训练样本数
LORA_RANK = 16
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 4                         # 7B 需改为 2
GRAD_ACCUM = 4

USE_QLORA = "7B" in MODEL_ID
MODEL_TAG = MODEL_ID.split('/')[-1]
print(f"模型: {MODEL_TAG}  |  QLoRA: {USE_QLORA}  |  样本: {MAX_SAMPLES}  |  epoch: {EPOCHS}")

## 1. 安装依赖

In [ ]:
!pip install -q transformers accelerate peft trl datasets
if USE_QLORA:
    !pip install -q bitsandbytes
print("依赖就绪 ✓")

## 2. 数据准备

使用开源中文指令数据集 alpaca-zh，统一转为 ChatML 格式。
这是实际工作中第一步：把原始数据处理成模型能吃的格式。

In [ ]:
from datasets import load_dataset

dataset = load_dataset("shibing624/alpaca-zh", split="train")
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

CHAT_TEMPLATE = """<|im_start|>user
{instruction}
<|im_end|>
<|im_start|>assistant
{output}
<|im_end|>"""

def format_chat(ex):
    return {"text": CHAT_TEMPLATE.format(instruction=ex["instruction"], output=ex["output"])}

dataset = dataset.map(format_chat)
print(f"训练数据: {len(dataset)} 条")
print(f"\n示例:")
print(dataset[0]["text"][:200])

## 3. 加载模型 + LoRA 配置

关键理解：LoRA 不修改原始权重，而是插入低秩矩阵。
这让你能用单卡 T4（16GB）微调大模型。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"加载 {MODEL_ID} ...")
if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"✓ 模型加载完成 | 参数量: {model.num_parameters()/1e9:.2f}B")
if USE_QLORA:
    print("  4-bit QLoRA 模式，显存占用约 6-8GB")

## 4. LoRA 训练

这是核心步骤。你会看到 loss 逐渐下降，说明模型在学。
重点关注 loss 数值的变化趋势，而不是绝对值。

In [ ]:
import time
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = f"./{MODEL_TAG}_lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    fp16=True,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=True,
)

print("开始训练 (训练中会每隔10步打印一次 loss)...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

final_loss = trainer.state.log_history[-1].get("train_loss", 0)
print(f"\n{'='*45}")
print(f"  训练完成")
print(f"  耗时: {elapsed/60:.1f} 分钟")
print(f"  最终 loss: {final_loss:.4f}")
print(f"  保存路径: {OUTPUT_DIR}")
print(f"{'='*45}")

## 5. 评估：微调前 vs 微调后

20 道技术问答，对比基座模型和微调模型的回答质量。
**这是面试时最能体现你思考的部分** — 你怎么评测一个模型的好坏？

In [ ]:
import json
from peft import PeftModel

# ===== 测试集（覆盖 AI 基础 / 工程实践 / 框架理解） =====
test_set = [
    "Transformer 中的多头注意力机制是如何工作的？",
    "什么是 KV Cache？在推理中起到什么作用？",
    "LoRA 微调的核心原理是什么？为什么比全量微调高效？",
    "什么是 RAG？它主要解决什么问题？",
    "ReAct 推理范式是什么？和 CoT 有什么区别？",
    "AWQ 量化和 GPTQ 量化有什么不同？",
    "Python 中装饰器是什么？写一个简单例子。",
    "什么是 Python GIL？它有什么影响？",
    "FastAPI 和 Flask 的主要区别是什么？",
    "Docker 中 COPY 和 ADD 指令有什么区别？",
    "什么是模型幻觉？有哪些缓解方法？",
    "LLM 推理时 temperature 参数的作用是什么？",
    "什么是 Prompt Engineering？有哪些常见技巧？",
    "LangChain 中 Chain 和 Agent 的核心区别是什么？",
    "什么是 PagedAttention？解决了什么问题？",
    "BN 和 LN 的区别是什么？为什么 NLP 中 LN 更常用？",
    "什么是梯度消失和梯度爆炸？如何缓解？",
    "Adam 优化器和 SGD 相比有什么优势？",
    "Fine-tuning 和 In-Context Learning 有什么区别？",
    "什么是 Tool Calling？LLM 如何决定调用哪个工具？",
]

# ===== 评分关键词（每道题 3-4 个核心概念） =====
keyword_map = {
    0: ["多个头", "拼接", "线性变换", "自注意力"],
    1: ["缓存", "Key", "Value", "加速", "自回归"],
    2: ["低秩", "冻结", "插入", "参数量"],
    3: ["检索", "增强", "生成", "知识库", "幻觉"],
    4: ["推理", "行动", "交替", "工具"],
    5: ["激活值", "缩放", "权重", "INT4"],
    6: ["高阶函数", "增强", "@", "动态"],
    7: ["全局锁", "线程", "CPython", "并行"],
    8: ["异步", "自动文档", "性能", "校验"],
    9: ["复制", "URL", "解压", "简洁"],
    10: ["事实错误", "RAG", "微调", "约束解码"],
    11: ["随机性", "采样", "多样性", "贪婪"],
    12: ["提示设计", "Few-shot", "CoT", "角色"],
    13: ["固定流程", "动态决策", "工具", "执行"],
    14: ["分页", "显存碎片", "KV", "虚拟内存"],
    15: ["Batch维度", "特征维度", "依赖batch", "NLP"],
    16: ["指数级", "残差", "梯度裁剪", "归一化"],
    17: ["自适应", "动量", "学习率", "收敛"],
    18: ["更新权重", "提示示例", "参数冻结", "灵活"],
    19: ["结构化", "JSON", "函数", "执行结果"],
}

def ask(model, tokenizer, question):
    prompt = f"<|im_start|>user\n{question}\n<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=96, temperature=0.1, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def hit_rate(answer, keywords):
    if not keywords:
        return 0
    hits = sum(1 for kw in keywords if kw.lower() in answer.lower())
    return hits / len(keywords)

# ===== 评估 =====
print("加载基座模型（微调前）...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
).eval()

print("加载微调后模型...")
lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
lora_model = PeftModel.from_pretrained(lora_model, OUTPUT_DIR).eval()

results = []
for i, q in enumerate(test_set):
    base_ans = ask(base, tokenizer, q)
    lora_ans = ask(lora_model, tokenizer, q)
    kws = keyword_map.get(i, [])
    base_score = hit_rate(base_ans, kws)
    lora_score = hit_rate(lora_ans, kws)
    results.append((q, base_ans, lora_ans, base_score, lora_score))

# ===== 输出对比 =====
print(f"\n{'='*70}")
print("  评估结果")
print(f"{'='*70}")
for i, (q, ba, la, bs, ls) in enumerate(results):
    diff = "↑" if ls > bs else ("↓" if ls < bs else "=")
    print(f"\n{i+1}. {q[:50]}")
    print(f"   基座: {ba[:60]}...")
    print(f"   微调: {la[:60]}...")
    print(f"   命中率: {bs:.0%} → {ls:.0%}  {diff}")

# ===== 汇总 =====
base_avg = sum(r[3] for r in results) / len(results)
lora_avg = sum(r[4] for r in results) / len(results)
improve = ((lora_avg - base_avg) / base_avg * 100) if base_avg > 0 else 0

better = sum(1 for r in results if r[4] > r[3])
worse = sum(1 for r in results if r[4] < r[3])

print(f"\n{'='*70}")
print(f"  汇总 ({len(results)} 题)")
print(f"{'='*70}")
print(f"  基座模型平均命中率:  {base_avg:.1%}")
print(f"  微调后平均命中率:  {lora_avg:.1%}")
print(f"  相对提升:          {improve:+.0f}%")
print(f"  微调更好/更差:      {better}/{worse}")
print(f"{'='*70}")

if lora_avg > base_avg:
    print("\n结论: LoRA 微调在该测试集上有效提升了回答质量 ✓")
else:
    print("\n结论: 微调后效果未明显提升。可能原因：训练数据与测试集领域不匹配、训练轮数不足。")

## 6.（可选）导出模型

保存到 Google Drive，方便后续使用或部署。

In [ ]:
from google.colab import drive
import shutil

try:
    drive.mount('/content/drive')
    save_path = f"/content/drive/MyDrive/{MODEL_TAG}_lora"
    shutil.copytree(OUTPUT_DIR, save_path, dirs_exist_ok=True)
    print(f"模型已保存到 Google Drive: {save_path}")
except Exception as e:
    print(f"保存失败（没关系，可以下次再存）: {e}")

## 7.（进阶）量化导出

合并 LoRA 权重 → GPTQ 量化 → 导出一个 INT4 模型文件。
**仅 1.5B 可在 Colab 完成**，7B 需在本地或有 GPU 的环境运行。

In [ ]:
if "1.5B" in MODEL_ID:
    # 合并权重
    print("合并 LoRA 权重...")
    lora_model = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
            device_map="auto", trust_remote_code=True),
        OUTPUT_DIR
    )
    merged = lora_model.merge_and_unload()
    merged.save_pretrained("./merged_model")
    tokenizer.save_pretrained("./merged_model")
    print("LoRA 合并完成 ✓")
    
    # GPTQ 量化
    !pip install -q optimum auto-gptq
    from optimum.gptq import GPTQQuantizer
    quantizer = GPTQQuantizer(bits=4, dataset="c4")
    quantized = quantizer.quantize_model(merged, tokenizer)
    quantized.save_pretrained("./qwen_int4")
    tokenizer.save_pretrained("./qwen_int4")
    print("GPTQ 量化完成 ✓ 已保存到 ./qwen_int4")
else:
    print("7B 量化需要更多显存，建议在本地 GPU 环境运行。")

---
## 完成

**你实际完成的工作**：

| 步骤 | 内容 | 面试能聊什么 |
|------|------|------------|
| ① 数据准备 | 指令数据格式化、清洗 | 为什么要统一 Chat Template、数据质量的影响 |
| ② LoRA 配置 | rank/lora_alpha/target_modules 选择 | 为什么选 rank=16、target_modules 怎么确定 |
| ③ 训练 | loss 收敛过程 | LoRA 训练注意事项、loss 多少算正常 |
| ④ 评估 | 20 道题前后对比 | 怎么设计评测、为什么用命中率而非 BLEU |
| ⑤ 量化（进阶）| GPTQ INT4 导出 | 量化原理、精度-速度 trade-off |

**这些就是大厂 LLM 应用开发岗位面试的核心话题**。